# AegisHunt Phase 4 Exploratory Analysis

This notebook reads tested library outputs. It does not implement conversion, splitting, leakage detection, model training, tuning, or evaluation. If no public benchmark has been manually acquired and validated, it uses the explicitly controlled synthetic demo. Demo results must not be presented as real-world performance.

In [ ]:
import csv
import json
from collections import Counter

from aegishunt.config import load_settings
from aegishunt.datasets.io import read_canonical_jsonl
from aegishunt.datasets.service import DatasetService

settings = load_settings().datasets
dataset_id = 'aegishunt-controlled-demo'
version = '1.0.0'
data_root = settings.processed_root / dataset_id / version
report_root = settings.reports_root / dataset_id / version
if not (data_root / 'canonical.jsonl').exists():
    DatasetService(settings).build_demo()
rows = read_canonical_jsonl(data_root / 'canonical.jsonl')

## Overview and provenance

In [ ]:
overview = {
    'dataset_id': rows[0].metadata.dataset_id,
    'dataset_version': rows[0].metadata.dataset_version,
    'controlled_synthetic': rows[0].metadata.provenance.get('synthetic') == 'true',
    'rows': len(rows),
    'groups': len({row.metadata.group_id for row in rows}),
    'source_files': len({row.metadata.source_file for row in rows}),
    'source_access_dates': sorted({row.metadata.source_access_date.isoformat() for row in rows}),
}
overview

## Class, family, group, and missing-value distributions

In [ ]:
distributions = {
    'binary': Counter(row.labels.binary_label for row in rows),
    'attack_family': Counter(row.labels.attack_family for row in rows),
    'group': Counter(row.metadata.group_id for row in rows),
    'missing_observed_at': sum(row.metadata.observed_at is None for row in rows),
}
distributions

## Quality, duplicate, feature range, and correlation-risk summaries

In [ ]:
quality = json.loads((report_root / 'quality_report.json').read_text())
leakage = json.loads((report_root / 'leakage_report.json').read_text())
split = json.loads((report_root / 'split_manifest.json').read_text())
with (report_root / 'feature_statistics.csv').open(newline='') as source:
    feature_statistics = list(csv.DictReader(source))
{
    'quality_status': quality['status'],
    'exact_duplicates': quality['exact_duplicate_count'],
    'near_duplicates': quality['near_duplicate_count'],
    'constant_features': quality['constant_features'],
    'missing_percentages': quality['missing_percentages'],
    'binary_class_percentages': quality['binary_class_percentages'],
    'attack_family_percentages': quality['attack_family_percentages'],
    'feature_distribution_statistics': feature_statistics,
    'leakage_status': leakage['status'],
    'correlation_warnings': leakage['correlation_warnings'],
    'unique_value_label_warnings': leakage['unique_value_label_warnings'],
    'split_rows': split['row_counts'],
    'split_groups': split['group_counts'],
    'split_families': split['attack_family_distributions'],
    'frozen_test': split['frozen_test'],
}

## Known limitations

The controlled demo uses harmless synthetic packet observations and the Phase 3 feature engine. It is not public benchmark data, captured enterprise traffic, or evidence of model quality. Public benchmark acquisition, checksum recording, label joining, and quality approval remain manual gates. The frozen test partition must not be used for feature, threshold, or model selection.